# Colab launcher

Runs the GPU work on Colab. This notebook holds no project logic: it prepares the VM and calls the
same commands as on a local machine, with `GEO_OVERLAYS=configs/colab.yaml` so every command uses
the Colab paths (data and caches on the VM's local disk, run outputs on Drive).

**Runtime → Change runtime type:** A100 for full fine-tunes; L4 is enough for the smoke runs and the
probe. Avoid T4 (no bfloat16) and TPUs.

**Colab secrets (key icon), notebook access on:** `GH_TOKEN` (GitHub, read, while the repo is
private) and `HF_TOKEN` (Hugging Face; DINOv3 access must be granted on the hub).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/geo-relations"
!mkdir -p {DRIVE}/runs {DRIVE}/hf_cache {DRIVE}/data_tarballs

In [ ]:
import os
from google.colab import userdata

REPO = "parsa-hke/veridiagram"  # GitHub owner/name
BRANCH = "main"
WORKDIR = "/content/veridiagram"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("GH_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.isdir(f"{WORKDIR}/.git"):
    !git -C {WORKDIR} pull --ff-only
else:
    !git clone -q -b {BRANCH} {url} {WORKDIR}
    !git -C {WORKDIR} remote set-url origin https://github.com/{REPO}.git  # drop token from .git/config
%cd {WORKDIR}
!git log -1 --oneline

In [ ]:
# Pinned versions (numpy/pandas match Colab's own packages). If pip asks for a restart, use
# Runtime -> Restart session, rerun the cells above (idempotent) and continue.
!pip install -q -r requirements.txt

In [ ]:
os.environ["HF_HOME"] = f"{DRIVE}/hf_cache"            # backbone weights survive sessions
os.environ["GEO_OVERLAYS"] = "configs/colab.yaml"       # every CLI uses the Colab paths
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
!python -m experiments.run --device-info

## Data

First session: downloads the datasets, builds everything (`python -m data.prepare`, ~15-20 min) and
saves one tarball to Drive. Later sessions: unpack that tarball to the local disk (~2 min). The
tarball name carries the git hash of the data pipeline, so a pipeline change rebuilds automatically.

In [ ]:
import subprocess
data_rev = subprocess.check_output(["git", "log", "-1", "--format=%h", "--", "data", "verifier", "configs/base.yaml"], text=True).strip()
tarball = f"{DRIVE}/data_tarballs/data_{data_rev}.tar"
print("data pipeline revision", data_rev)
if os.path.exists(tarball):
    !mkdir -p /content/data && tar -C /content -xf {tarball}
else:
    !python -m data.prepare --download
    !tar -C /content -cf {tarball} data
!du -sh /content/data/raw /content/data/processed

## Phase 6 gate: smoke runs (must pass before any full run)

In [ ]:
# DINOv3 access check: loads the gated checkpoint with NO fallback; must print OK.
!python -m models.backbone
!python -m experiments.run --config configs/probe.yaml --smoke
!python -m experiments.run --config configs/finetune.yaml --smoke
!tail -n 2 {DRIVE}/runs/log.csv

## Full runs

Results go to `MyDrive/geo-relations/runs/<name>_s<seed>/` and `runs/log.csv`. A fine-tune that is
interrupted resumes from its last epoch when the same command is run again.

In [ ]:
CONFIG = "configs/finetune.yaml"
for seed in [0, 1, 2]:
    !python -m experiments.run --config {CONFIG} --set seed={seed}